# 10 · Sinkhorn-Knopp, hyper-connections and mHC

**Needed for:** Hyper-Connections (Zhu et al., 2024) and DeepSeek's mHC (arXiv 2512.24880), LLM track step 8.
mHC is used in DeepSeek-V4.

A standard residual stream is one vector per token: $x_{l+1} = x_l + F(x_l)$. **Hyper-connections** widen it to
$n$ parallel streams (an $n \times C$ matrix per token) and learn how to mix them:

$$x_{l+1} = H^{res}_l\, x_l + H^{post\top}_l\, F\big(H^{pre}_l\, x_l\big)$$

- $H^{pre}$ ($1\times n$): how to combine the streams into the layer's input
- $H^{post}$ ($1\times n$): how to write the layer's output back into each stream
- $H^{res}$ ($n\times n$): how streams mix with each other on the skip path

The problem: across $L$ layers the skip path multiplies $H^{res}_L \cdots H^{res}_1$. Unconstrained, that product
can blow up or vanish, which is the instability plain residual connections were invented to avoid.
**mHC's fix:** force each $H^{res}$ to be **doubly stochastic** (non-negative, every row and column sums to 1),
using the Sinkhorn-Knopp algorithm. Products of doubly stochastic matrices stay doubly stochastic, so the skip
path can only average streams, never amplify them.

This notebook uses static (input-independent) coefficients. In the paper they are computed from the input,
with learned static biases; check the paper for the exact parameterization.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. Sinkhorn-Knopp

Start from any positive matrix ($e^{\text{logits}}$). Alternately divide each row by its sum and each column by
its sum. It converges to a doubly stochastic matrix. Every step is differentiable, so gradients flow back to
the logits. Iterating in log space (logsumexp) avoids overflow; mHC uses about 20 iterations.

In [ ]:
def sinkhorn(logits, n_iters=20):
    """logits: (..., n, n) -> doubly stochastic (..., n, n)."""
    log_p = logits
    for _ in range(n_iters):
        log_p = log_p - log_p.logsumexp(dim=-1, keepdim=True)   # rows sum to 1
        log_p = log_p - log_p.logsumexp(dim=-2, keepdim=True)   # columns sum to 1
    return log_p.exp()


def test_sinkhorn(fn):
    torch.manual_seed(0)
    logits = torch.randn(3, 4, 4) * 3
    P = fn(logits, n_iters=50)
    assert (P >= 0).all()
    torch.testing.assert_close(P.sum(-1), torch.ones(3, 4), atol=1e-4, rtol=0)
    torch.testing.assert_close(P.sum(-2), torch.ones(3, 4), atol=1e-4, rtol=0)
    logits.requires_grad_(True)
    fn(logits).sum(-1)[..., 0].sum().backward()
    assert logits.grad is not None and torch.isfinite(logits.grad).all()


test_sinkhorn(sinkhorn)
print(sinkhorn(torch.randn(4, 4)).round(decimals=3))

## 2. Why the constraint matters: products over depth

Multiply $L$ mixing matrices, each close to the identity, and track the largest singular value (how much the skip
path can amplify a signal). A doubly stochastic matrix has spectral norm exactly 1, and so does any product of them.

In [ ]:
n = 4
for L in (1, 8, 32, 64):
    prod_free, prod_ds = torch.eye(n), torch.eye(n)
    torch.manual_seed(1)
    for _ in range(L):
        logits = torch.eye(n) * 2 + 0.3 * torch.randn(n, n)
        prod_free = (torch.eye(n) + 0.3 * torch.randn(n, n)) @ prod_free   # unconstrained, near identity
        prod_ds = sinkhorn(logits) @ prod_ds
    print(f"depth {L:3d}: unconstrained gain {torch.linalg.matrix_norm(prod_free, ord=2).item():10.2f}"
          f"   doubly stochastic gain {torch.linalg.matrix_norm(prod_ds, ord=2).item():.4f}")

## 3. A hyper-connection block

The residual state per token is now $(n, C)$. Initialize so the block behaves like an ordinary residual block:
$H^{res} \approx I$, $H^{pre}$ reads mostly one stream, $H^{post}$ writes to all streams.
At the start of the network, copy the embedding into all $n$ streams; at the end, sum them.

In [ ]:
def hc_forward(x, H_res, h_pre, h_post, layer):
    """Functional form for the drill. x: (B, T, n, C), H_res: (n, n), h_pre/h_post: (n,)."""
    layer_in = torch.einsum("n,btnc->btc", h_pre, x)
    return torch.einsum("mn,btnc->btmc", H_res, x) + h_post[None, None, :, None] * layer(layer_in)[:, :, None, :]


class HyperConnection(nn.Module):
    def __init__(self, layer, n_streams, constrained=True):
        super().__init__()
        self.layer, self.n, self.constrained = layer, n_streams, constrained
        if constrained:   # parameters are logits; mixing() maps them onto the constraint set
            self.res = nn.Parameter(torch.eye(n_streams) * 4.0)          # near-identity after Sinkhorn
            self.pre = nn.Parameter(torch.zeros(n_streams))
            self.post = nn.Parameter(torch.zeros(n_streams))
        else:             # plain hyper-connections: parameters are used as-is
            self.res = nn.Parameter(torch.eye(n_streams))
            self.pre = nn.Parameter(torch.full((n_streams,), 1.0 / n_streams))
            self.post = nn.Parameter(torch.ones(n_streams))

    def mixing(self):
        if self.constrained:   # mHC: doubly stochastic residual mixing, non-negative pre/post weights
            return sinkhorn(self.res), torch.sigmoid(self.pre), torch.sigmoid(self.post)
        return self.res, self.pre, self.post

    def forward(self, x):
        """x: (B, T, n, C)."""
        H_res, h_pre, h_post = self.mixing()
        return hc_forward(x, H_res, h_pre, h_post, self.layer)


def test_hc_forward(fn):
    torch.manual_seed(0)
    B, T, n, C = 2, 3, 4, 8
    x = torch.randn(B, T, n, C)
    layer = nn.Linear(C, C)
    # with one stream and identity mixing, a hyper-connection is an ordinary residual block
    x1 = x[:, :, :1]
    out1 = fn(x1, torch.eye(1), torch.ones(1), torch.ones(1), layer)
    torch.testing.assert_close(out1[:, :, 0], x1[:, :, 0] + layer(x1[:, :, 0]))
    H, pre, post = sinkhorn(torch.randn(n, n)), torch.rand(n), torch.rand(n)
    ref = torch.stack([sum(H[m, j] * x[:, :, j] for j in range(n)) + post[m] * layer(sum(pre[j] * x[:, :, j] for j in range(n)))
                       for m in range(n)], dim=2)
    torch.testing.assert_close(fn(x, H, pre, post, layer), ref)


test_hc_forward(hc_forward)
print("hyper-connection forward ok")

## 4. A deep stack: activation growth

64 blocks of a small MLP. Add the same small random noise to every mixing parameter, mimicking what training
might do, and compare how big the residual state gets.

In [ ]:
def deep_stack_norm(constrained, depth=64, n=4, C=64, noise=0.2):
    torch.manual_seed(0)
    blocks = []
    for _ in range(depth):
        mlp = nn.Sequential(nn.LayerNorm(C), nn.Linear(C, C), nn.GELU(), nn.Linear(C, C))
        hc = HyperConnection(mlp, n, constrained)
        with torch.no_grad():
            for p in (hc.res, hc.pre, hc.post):
                p.add_(noise * torch.randn_like(p))   # same noise size in both parameterizations
        blocks.append(hc)
    x = torch.randn(8, 16, C)[:, :, None].expand(8, 16, n, C)
    norms = []
    with torch.no_grad():
        for i, b in enumerate(blocks):
            x = b(x)
            if (i + 1) in (1, 16, 32, 64):
                norms.append(round(x.norm(dim=-1).mean().item(), 1))
    return norms


print("mean residual norm after 1/16/32/64 blocks")
print("  unconstrained HC:", deep_stack_norm(constrained=False))
print("  mHC (Sinkhorn):  ", deep_stack_norm(constrained=True))

## Things to be able to explain

- Why is a doubly stochastic matrix's spectral norm at most 1? (It's a convex combination of permutation
  matrices, each with norm 1 — Birkhoff–von Neumann.)
- What does $n$ streams cost? Memory traffic for the residual state grows $n\times$; mHC's paper spends a
  lot of effort on kernel fusion and recomputation to keep the overhead small.
- Why non-negative $H^{pre}$ and $H^{post}$? (Mixing with signs can cancel streams; non-negative weights keep it
  an averaging operation.)

## Drill

In [ ]:
def my_sinkhorn(logits, n_iters=20):
    raise NotImplementedError


def my_hc_forward(x, H_res, h_pre, h_post, layer):
    raise NotImplementedError


drill(test_sinkhorn, my_sinkhorn)
drill(test_hc_forward, my_hc_forward)